<a href="https://colab.research.google.com/github/dhanushiyad31/calculator-/blob/main/DCGAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"dhanu31d","key":"923662075734f7694c9211e828e85a60"}'}

In [ ]:
import os, glob, math, random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from torchvision.utils import save_image, make_grid

# --------------------
# Config
# --------------------
class Cfg:
    data_dir = "/content/sample_data/data/train"       # folder with images (jpg/png/jpeg)
    out_dir = "/content/sample_data/ganruns"       # where to save samples & weights
    img_size = 64                # 64x64 DCGAN
    channels = 3                 # 3 for RGB
    z_dim = 128                  # latent vector size
    g_feat = 32                  # generator feature width
    d_feat = 32                  # discriminator feature width
    batch_size = 16
    num_workers = 4
    epochs = 180         # increase as you add data
    lr = 3e-4
    beta1 = 0.5
    beta2 = 0.999
    sample_every = 200     # iterations
    device = "cuda" if torch.cuda.is_available() else "cpu"
    aug = True                   # enable strong augmentations

cfg = Cfg()
Path(cfg.out_dir, "samples").mkdir(parents=True, exist_ok=True)

# --------------------
# Dataset
# --------------------
class ImageFolderDataset(Dataset):
    def __init__(self, root, img_size=64, channels=3, aug=True):
        exts = ("*.jpg","*.jpeg","*.png","*.bmp","*.tif","*.tiff")
        self.files = []
        for e in exts:
            self.files += glob.glob(os.path.join(root, e))
        if len(self.files) == 0:
            raise RuntimeError(f"No images found in {root}")

        t_aug = []
        if aug:
            # With few images, heavy aug helps: flips, small rotations, color jitter, random crop
            t_aug += [
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomApply([transforms.ColorJitter(0.2,0.2,0.2,0.05)], p=0.8),
                transforms.RandomAffine(degrees=8, translate=(0.02,0.02), scale=(0.95,1.05)),
                transforms.RandomResizedCrop(img_size, scale=(0.9,1.0), ratio=(0.9,1.1)),
            ]
        else:
            t_aug += [transforms.Resize((img_size, img_size))]

        self.tf = transforms.Compose([
            *t_aug,
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*channels, [0.5]*channels)
        ])
        self.channels = channels

    def __len__(self):
        # Repeat virtually to provide enough steps per epoch with tiny datasets
        return len(self.files) * 200

    def __getitem__(self, idx):
        real_idx = idx % len(self.files)
        img = Image.open(self.files[real_idx]).convert("RGB")
        return self.tf(img)

# --------------------
# DCGAN models
# --------------------
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.zeros_(m.bias.data)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.zeros_(m.bias.data)

class Generator(nn.Module):
    def __init__(self, z_dim, g_feat, channels):
        super().__init__()
        self.net = nn.Sequential(
            # input Z: (z_dim) -> (g_feat*8) x 4 x 4
            nn.ConvTranspose2d(z_dim, g_feat*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(g_feat*8), nn.ReLU(True),

            # 4x4 -> 8x8
            nn.ConvTranspose2d(g_feat*8, g_feat*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(g_feat*4), nn.ReLU(True),

            # 8x8 -> 16x16
            nn.ConvTranspose2d(g_feat*4, g_feat*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(g_feat*2), nn.ReLU(True),

            # 16x16 -> 32x32
            nn.ConvTranspose2d(g_feat*2, g_feat, 4, 2, 1, bias=False),
            nn.BatchNorm2d(g_feat), nn.ReLU(True),

            # 32x32 -> 64x64
            nn.ConvTranspose2d(g_feat, channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, d_feat, channels):
        super().__init__()
        self.net = nn.Sequential(
            # 64x64 -> 32x32
            nn.Conv2d(channels, d_feat, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # 32x32 -> 16x16
            nn.Conv2d(d_feat, d_feat*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(d_feat*2),
            nn.LeakyReLU(0.2, inplace=True),

            # 16x16 -> 8x8
            nn.Conv2d(d_feat*2, d_feat*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(d_feat*4),
            nn.LeakyReLU(0.2, inplace=True),

            # 8x8 -> 4x4
            nn.Conv2d(d_feat*4, d_feat*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(d_feat*8),
            nn.LeakyReLU(0.2, inplace=True),

            # 4x4 -> 1
            nn.Conv2d(d_feat*8, 1, 4, 1, 0, bias=False)
        )
    def forward(self, x):
        return self.net(x).view(-1)

# --------------------
# Training
# --------------------
def train():
    ds = ImageFolderDataset(cfg.data_dir, cfg.img_size, cfg.channels, cfg.aug)
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True,
                    num_workers=cfg.num_workers, drop_last=True, pin_memory=True)

    G = Generator(cfg.z_dim, cfg.g_feat, cfg.channels).to(cfg.device)
    D = Discriminator(cfg.d_feat, cfg.channels).to(cfg.device)
    G.apply(weights_init); D.apply(weights_init)

    optG = optim.Adam(G.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))
    optD = optim.Adam(D.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))

    fixed_z = torch.randn(64, cfg.z_dim, 1, 1, device=cfg.device)

    step = 0
    for epoch in range(cfg.epochs):
        for real in dl:
            real = real.to(cfg.device)

            # ---------------- D ----------------
            z = torch.randn(real.size(0), cfg.z_dim, 1, 1, device=cfg.device)
            with torch.no_grad():
                fake = G(z)

            D_real = D(real)
            D_fake = D(fake.detach())

            # Hinge loss
            d_loss = (F.relu(1. - D_real).mean() + F.relu(1. + D_fake).mean())

            optD.zero_grad()
            d_loss.backward()
            optD.step()

            # ---------------- G ----------------
            z2 = torch.randn(real.size(0), cfg.z_dim, 1, 1, device=cfg.device)
            fake2 = G(z2)
            g_loss = -D(fake2).mean()

            optG.zero_grad()
            g_loss.backward()
            optG.step()

            # ---------------- Logging / Samples ----------------
            if step % cfg.sample_every == 0:
                G.eval()
                with torch.no_grad():
                    samples = G(fixed_z).cpu()
                grid = make_grid(samples, nrow=8, normalize=True, value_range=(-1,1))
                save_image(grid, f"{cfg.out_dir}/samples/step_{step:07d}.png")
                G.train()

            if step % 50 == 0:
                print(f"Epoch {epoch+1}/{cfg.epochs} | step {step} | D: {d_loss.item():.3f} | G: {g_loss.item():.3f}")

            step += 1

        # save checkpoints per epoch
        torch.save(G.state_dict(), f"{cfg.out_dir}/G.pt")
        torch.save(D.state_dict(), f"{cfg.out_dir}/D.pt")

if __name__ == "__main__":
    print(f"Using device: {cfg.device}")
    train()

Using device: cuda


RuntimeError: No images found in /content/sample_data/data/train